In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import openai
import json
from threading import Lock
import pandas as pd
import re
import time
import nltk
import spacy
import numpy as np
from collections import Counter

# Load English tokenizer, POS tagger, parser, NER, and word vectors
nlp = spacy.load("en_core_web_sm")

# Define a word list using nltk corpus
english_vocab = set(w.lower() for w in nltk.corpus.words.words())

open_ai_keys = [
    os.getenv("OPENAI_KEY_1"),
    os.getenv("OPENAI_KEY_2"),
    os.getenv("OPENAI_KEY_3"),
    os.getenv("OPENAI_KEY_4"),
    os.getenv("OPENAI_KEY_5"),
    os.getenv("OPENAI_KEY_6"),
    os.getenv("OPENAI_KEY_7"),
    os.getenv("OPENAI_KEY_8"),
    os.getenv("OPENAI_KEY_9"),
    os.getenv("OPENAI_KEY_10"),
    os.getenv("OPENAI_KEY_11"),
    os.getenv("OPENAI_KEY_12"),
    os.getenv("OPENAI_KEY_13"),
    os.getenv("OPENAI_KEY_14"),
    os.getenv("OPENAI_KEY_15"),
    os.getenv("OPENAI_KEY_16"),
    os.getenv("OPENAI_KEY_17"),
    os.getenv("OPENAI_KEY_18"),
    os.getenv("OPENAI_KEY_19"),
    os.getenv("OPENAI_KEY_20"),
    os.getenv("OPENAI_KEY_21"),
    os.getenv("OPENAI_KEY_22"),
    os.getenv("OPENAI_KEY_23"),
    os.getenv("OPENAI_KEY_24"),
    os.getenv("OPENAI_KEY_25")
]

executor = ThreadPoolExecutor(max_workers=25)

transcript = "../runs/2008-09-26-debate-preclean.txt"

def process_transcript(file_path):
    with open(file_path, "r") as file:
        content = file.read()

    lines = content.split("\n")
    current_speaker = None
    turn = 0
    speaker_turn_count = Counter()
    rows = []
    current_line = ""

    for line in lines:
        if ": " in line:
            speaker, line = line.split(": ", 1)
            if speaker != current_speaker:
                if current_speaker is not None:
                    rows.append([turn, current_speaker, current_line.strip(), speaker_turn_count[current_speaker]])
                turn += 1
                current_speaker = speaker
                speaker_turn_count[speaker] += 1
                current_line = line
            else:
                current_line += " " + line
        else:
            current_line += " " + line

    rows.append([turn, current_speaker, current_line.strip(), speaker_turn_count[current_speaker]])

    df = pd.DataFrame(rows, columns=["Turn", "Speaker", "Content", "Cumulative_Speaker_Turn"])

    return df

transcript_dataframe = process_transcript(transcript)

def make_scoreboard(df):
    df['Base_Score'] = 0
    df['EPL_Multiplier'] = 0.0
    df['Total_Score'] = 0.0
    df['Sentence_Count'] = 0
    df['Word_Count'] = 0
    df['Word_Count_Bonus'] = 0
    df['Named_Entity_Count'] = 0
    df['E'] = 0.0
    df['P'] = 0.0
    df['L'] = 0.0
    df['E_Expl'] = "None"
    df['P_Expl'] = "None"
    df['L_Expl'] = "None"
    df['Cumulative_Speaker_Base_Score'] = 0
    df['Cumulative_Speaker_Total_Score'] = 0.0
    df['Cumulative_Speaker_Turn'] = 0
    
    return df

empty_scoreboard = make_scoreboard(transcript_dataframe)

def calculate_scores(df):
    df['Sentence_Count'] = df['Content'].apply(lambda x: len(nltk.sent_tokenize(x)))
    df['Word_Count'] = df['Content'].apply(lambda x: len(nltk.word_tokenize(x)))
    df['Word_Count_Bonus'] = df['Content'].apply(lambda x: len(set(w.lower() for w in nltk.word_tokenize(x) if w.lower() in english_vocab)))
    df['Named_Entity_Count'] = df['Content'].apply(lambda x: len(nlp(x).ents))
    
    df['Base_Score'] = df['Sentence_Count'] + df['Word_Count'] + df['Word_Count_Bonus'] + df['Named_Entity_Count']
    df['Total_Score'] = df['Base_Score']  # Initially Total_Score is the same as Base_Score

    # Calculate EPL Multiplier based on current E, P, L scores
    df['EPL_Multiplier'] = (df['E'].astype(float) + df['P'].astype(float) + df['L'].astype(float)) / 30
    
    df['Cumulative_Speaker_Base_Score'] = df.groupby('Speaker')['Base_Score'].cumsum()
    df['Cumulative_Speaker_Total_Score'] = df.groupby('Speaker')['Total_Score'].cumsum() 
    df['Cumulative_Speaker_Turn'] = df.groupby('Speaker').cumcount() + 1

    return df
# Calculate the initial scores
scoreboard = calculate_scores(empty_scoreboard)

def parse_content(content):
    try:
        # Try to parse the string as a JSON
        parsed_content = json.loads(content.replace("'", "\""))  # replace single quotes with double quotes
    except json.JSONDecodeError:
        # If it fails, parse it using regular expressions
        score_matches = re.findall(r'(\w+_?\w*): ([\d\.]+|".*?")(?=, \w+_?\w*:|$)', content)
        parsed_content = {key: value.strip('""') for key, value in score_matches}

        # Parse explanations
        explanation_matches = re.findall(r'(\w+_Expl): ("[^"]*")(?=, \w+_Expl:|$)', content)
        parsed_explanations = {}
        for key, value in explanation_matches:
            try:
                parsed_explanations[key] = json.loads(value)
            except json.JSONDecodeError:
                # If it fails to load as JSON (which will be the case for an empty string), just use the original value
                parsed_explanations = {key: value.strip('""') for key, value in explanation_matches}

        # Combine the parsed scores and explanations
        parsed_content.update(parsed_explanations)

    return parsed_content

def prompt_epl(row, prev_row=None):
    if not row['Content']: # Ensure that there is a current sentence
        return "Error: A current sentence is required."

    # Initialize content with the current sentence
    content = f"Evaluate the following snippet for Ethos (E), Pathos (P), and Logos (L). (Scale: 0.0 - 9.9) For non-zero score only: provide a brief explanation, otherwise leave empty string. It should be equally difficult to score 0.0 as it would be to score 9.9. [Snippet: \'{row['Content']}\'] Return only a JSON object in this schema: {{E: x.x, P: y.y, L: z.z, E_Expl: string, P_Expl:, string L_Expl: string}}. Explanations must be 1 sentence long."

    prompt = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[
            {
                "role": "system",
                "content": "You are a master debater and assistant to the judge of the US Presidential Debates."
            },
            {
                "role": "user",
                "content": content
            }
        ]
    )
    return prompt

def threadlock(idx, output_file_full, df, lock1, lock2, api_key):
    retries = 5
    for attempt in range(retries):
        try:
            # Get current row and previous row (if exists)
            row = df.loc[idx]
            prev_row = df.loc[idx - 1] if idx > 0 else None

            # Set the API key
            openai.api_key = api_key

            # Call OpenAI API and get response
            response = prompt_epl(row, prev_row)

            # Write the complete response to the JSON file
            with lock1:
                with open(output_file_full, 'a') as f:
                    json.dump(response, f)
                    f.write('\n')

            # Extract 'content' field and parse it to get scores and explanations
            content = response['choices'][0]['message']['content']
            parsed_content = parse_content(content)

            # Update DataFrame with new values (with thread lock to avoid race conditions)
            lock2.acquire()
            try:
                for key, value in parsed_content.items():
                    df.loc[idx, key] = value
                
                # Update EPL Multiplier and Total Score columns
                df.loc[idx, 'EPL_Multiplier'] = 1 + ((float(df.loc[idx, 'E']) + float(df.loc[idx, 'P']) + float(df.loc[idx, 'L'])) / 30)  # Adjusted the formula here
                df.loc[idx, 'Base_Score'] = float(df.loc[idx, 'Base_Score'])  # Explicit conversion to float
                df.loc[idx, 'Total_Score'] = df.loc[idx, 'Base_Score'] * (1 + (df.loc[idx, 'EPL_Multiplier']))
     
                # Update Cumulative_Speaker_Base_Score and Cumulative_Speaker_Total_Score
                df.loc[idx, 'Cumulative_Speaker_Base_Score'] = df[df['Speaker'] == df.loc[idx, 'Speaker']]['Base_Score'].sum()
                df.loc[idx, 'Cumulative_Speaker_Total_Score'] = df[df['Speaker'] == df.loc[idx, 'Speaker']]['Total_Score'].sum()

            finally:
                lock2.release()

            # If we've reached this point, it means there were no exceptions, so we can break the loop
            break

        except Exception as e:
            # If it's the last attempt and it still fails, we print the error and move on to the next task
            if attempt == retries - 1:
                print(f'Error at index {idx} after {retries} attempts: {str(e)}')
                print(f'Response content: {response["choices"][0]["message"]["content"]}')  # Add this line
            else:
                # If it's not the last attempt, we wait for a bit and try again
                time.sleep(2 ** attempt)

def get_epl(input_dataframe, output_file_full, startturn=None, endturn=None, runcount=None):
    # Error checking for parameters
    if startturn is not None and endturn is not None:
        if runcount is not None:
            raise ValueError("Parameter 'runcount' cannot be used together with 'startturn' and 'endturn'. Use 'runcount' alone or with either 'startturn' and 'endturn'.")

    # Determine start and end points for analysis
    start = startturn if startturn is not None else 0
    end = endturn if endturn is not None else len(input_dataframe)

    # Determine number of runs
    if runcount is not None:
        end = min(start + runcount, len(input_dataframe))

    # Check if 'end' exceeds the length of the speaker turns
    if end > len(input_dataframe):
        end = len(input_dataframe)
        print(f"Warning: The end point exceeded the length of the dataframe. It has been adjusted to the last turn: {end}")

    # Create locks for writing to file and dataframe
    lock1 = Lock()
    lock2 = Lock()

    # Use a ThreadPoolExecutor to process the lines concurrently
    with ThreadPoolExecutor(max_workers=25) as executor:
        # Assign API keys cyclically and start all the tasks
        futures = [executor.submit(threadlock, i, output_file_full, input_dataframe, lock1, lock2, open_ai_keys[i % len(open_ai_keys)]) for i in range(start, end)]
        
        # Ensure all threads have finished
        for future in as_completed(futures):
            try:
                result = future.result()
            except Exception as e:
                print(f"Error in a thread: {str(e)}")

    # Sort DataFrame by 'Turn' after all threads are completed
    input_dataframe.sort_values('Turn', inplace=True)
    input_dataframe.reset_index(drop=True, inplace=True)

    # Reset openai.api_key to None after the job is done
    openai.api_key = None

    return input_dataframe

In [16]:
scoreboard.loc[300:307]


,Turn,Speaker,Content,Cumulative_Speaker_Turn,Base_Score,EPL_Multiplier,Total_Score,Sentence_Count,Word_Count,Word_Count_Bonus,Named_Entity_Count,E,P,L,E_Expl,P_Expl,L_Expl,Cumulative_Speaker_Base_Score,Cumulative_Speaker_Total_Score
300,301,MCCAIN,"First of all, I won’t repeat the mistake that ...",93,820,0.0,820.0,31,590,161,38,0.0,0.0,0.0,None,None,None,19005,19005.0
301,302,OBAMA,Nobody talked about attacking Pakistan. Here’s...,94,526,0.0,526.0,17,365,122,22,0.0,0.0,0.0,None,None,None,20161,20161.0
302,303,MCCAIN,I — I don’t think that Senator Obama understan...,94,174,0.0,174.0,4,111,50,9,0.0,0.0,0.0,None,None,None,19179,19179.0
303,304,"And I saw that, and I saw the situation, and I...",Nearly 300 Marines lost their lives in the bom...,2,589,0.0,589.0,16,401,146,26,0.0,0.0,0.0,None,None,None,1178,1178.0
304,305,MCCAIN,"A war that I was in, where we had an Army, tha...",95,134,0.0,134.0,3,81,48,2,0.0,0.0,0.0,None,None,None,19313,19313.0
305,306,OBAMA,"Jim, let me just make a point. I’ve got a brac...",95,440,0.0,440.0,14,289,120,17,0.0,0.0,0.0,None,None,None,20601,20601.0
306,307,LEHRER,New …,103,4,0.0,4.0,1,2,1,0,0.0,0.0,0.0,None,None,None,4455,4455.0
307,308,MCCAIN,You might think that with that kind of concern...,96,275,0.0,275.0,7,177,79,12,0.0,0.0,0.0,None,None,None,19588,19588.0


In [7]:
scored_scoreboard = get_epl(scoreboard, "2008-test-23.json", startturn=1, endturn=5)

In [10]:
import json

def parse_responses_and_write_json(input_filename, output_filename):
    parsed_contents = []

    # Open and read the JSON file line by line
    with open(input_filename, 'r') as file:
        for line in file:
            response = json.loads(line)
            
            # Parse the content portion
            content = json.loads(response['choices'][0]['message']['content'])
            parsed_contents.append(content)

    # Write to a new JSON file with indentation
    with open(output_filename, 'w') as f:
        json.dump(parsed_contents, f, indent=4)

# Usage
parse_responses_and_write_json('2008-test-22.json', 'parsed_contents-2.json')